# [[9, 1, 3]] Shor Code

## Introduction

The Shor code, introduced by Peter Shor in 1995, was the first quantum error correcting code. It is a concatenated code that combines the properties of the 3-qubit bit-flip code and the 3-qubit phase-flip code to protect against arbitrary single-qubit errors (X, Y or Z).

### Construction Principle

In [17]:
# Imports and Setup
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

print("Imports successful. Simulator backend ready.")


Imports successful. Simulator backend ready.


In [18]:
# Cell 2: Define the Shor Code Class
class ShorCode:
    def __init__(self):
        # 9 Data Qubits
        self.qr_data = QuantumRegister(9, name='data')
        # 8 Ancilla Qubits
        self.qr_anc = QuantumRegister(8, name='anc')

        # Classical Registers for Syndromes
        self.cr_z1 = ClassicalRegister(2, name='syn_z1') # Block 1
        self.cr_z2 = ClassicalRegister(2, name='syn_z2') # Block 2
        self.cr_z3 = ClassicalRegister(2, name='syn_z3') # Block 3
        self.cr_x  = ClassicalRegister(2, name='syn_x')  # Phase

        # Final readout
        self.cr_out = ClassicalRegister(9, name='readout')

        # Initialize Circuit
        self.qc = QuantumCircuit(
            self.qr_data, self.qr_anc,
            self.cr_z1, self.cr_z2, self.cr_z3, self.cr_x, self.cr_out
        )

    def encode(self):
        """
        Encodes a single logical |0> into the 9-qubit Shor state.
        """
        # 1. Phase-flip protection (Outer Layer) -> |+++>
        self.qc.cx(self.qr_data[0], self.qr_data[3])
        self.qc.cx(self.qr_data[0], self.qr_data[6])
        self.qc.h(self.qr_data[0])
        self.qc.h(self.qr_data[3])
        self.qc.h(self.qr_data[6])

        # 2. Bit-flip protection (Inner Layer) -> GHZ states
        # Block 1
        self.qc.cx(self.qr_data[0], self.qr_data[1])
        self.qc.cx(self.qr_data[0], self.qr_data[2])
        # Block 2
        self.qc.cx(self.qr_data[3], self.qr_data[4])
        self.qc.cx(self.qr_data[3], self.qr_data[5])
        # Block 3
        self.qc.cx(self.qr_data[6], self.qr_data[7])
        self.qc.cx(self.qr_data[6], self.qr_data[8])

        self.qc.barrier(label='Encoding Done')

    def inject_error(self, error_type, qubit_index):
        """
        Injects a specific error for testing.
        error_type: 'X', 'Y', or 'Z'
        """
        if error_type == 'X':
            self.qc.x(self.qr_data[qubit_index])
        elif error_type == 'Z':
            self.qc.z(self.qr_data[qubit_index])
        elif error_type == 'Y':
            self.qc.y(self.qr_data[qubit_index])

        self.qc.barrier(label=f'Error {error_type}_{qubit_index}')

    def measure_syndromes(self):
        """
        Non-destructive measurement of the 8 stabilizers using ancillas.
        """
        # --- Z-Stabilizers (Bit Flips) ---
        # We check parity of neighbors (q0-q1, q1-q2, etc.)
        # Logic: CNOT(data -> ancilla)

        # Block 1 (Ancillas 0, 1)
        self.qc.cx(self.qr_data[0], self.qr_anc[0])
        self.qc.cx(self.qr_data[1], self.qr_anc[0])
        self.qc.cx(self.qr_data[1], self.qr_anc[1])
        self.qc.cx(self.qr_data[2], self.qr_anc[1])

        # Block 2 (Ancillas 2, 3)
        self.qc.cx(self.qr_data[3], self.qr_anc[2])
        self.qc.cx(self.qr_data[4], self.qr_anc[2])
        self.qc.cx(self.qr_data[4], self.qr_anc[3])
        self.qc.cx(self.qr_data[5], self.qr_anc[3])

        # Block 3 (Ancillas 4, 5)
        self.qc.cx(self.qr_data[6], self.qr_anc[4])
        self.qc.cx(self.qr_data[7], self.qr_anc[4])
        self.qc.cx(self.qr_data[7], self.qr_anc[5])
        self.qc.cx(self.qr_data[8], self.qr_anc[5])

        # --- X-Stabilizers (Phase Flips) ---
        # Logic: Hadamard(anc) -> CNOT(anc -> data) -> Hadamard(anc)
        self.qc.h(self.qr_anc[6])
        self.qc.h(self.qr_anc[7])

        # Ancilla 6 checks Block 1 vs Block 2 (X0...X5)
        for i in range(0, 6):
            self.qc.cx(self.qr_anc[6], self.qr_data[i])

        # Ancilla 7 checks Block 2 vs Block 3 (X3...X8)
        for i in range(3, 9):
            self.qc.cx(self.qr_anc[7], self.qr_data[i])

        self.qc.h(self.qr_anc[6])
        self.qc.h(self.qr_anc[7])

        self.qc.barrier(label='Measure Syndromes')

        # --- Measurements ---
        # Measure Z-checks
        self.qc.measure(self.qr_anc[0], self.cr_z1[0])
        self.qc.measure(self.qr_anc[1], self.cr_z1[1])
        self.qc.measure(self.qr_anc[2], self.cr_z2[0])
        self.qc.measure(self.qr_anc[3], self.cr_z2[1])
        self.qc.measure(self.qr_anc[4], self.cr_z3[0])
        self.qc.measure(self.qr_anc[5], self.cr_z3[1])

        # Measure X-checks
        self.qc.measure(self.qr_anc[6], self.cr_x[0])
        self.qc.measure(self.qr_anc[7], self.cr_x[1])

    def apply_correction(self):
        """
        Uses dynamic circuits (if_test) to apply corrections based on syndromes.
        """

        # --- Correcting Bit Flips (X) ---
        # Syndrome '01' (dec 1) -> Error on Qubit 0 (Leftmost of block)
        # Syndrome '10' (dec 2) -> Error on Qubit 2 (Rightmost of block)
        # Syndrome '11' (dec 3) -> Error on Qubit 1 (Middle of block)

        # Block 1
        with self.qc.if_test((self.cr_z1, 1)): self.qc.x(self.qr_data[0]) # Correct Q0
        with self.qc.if_test((self.cr_z1, 2)): self.qc.x(self.qr_data[2]) # Correct Q2
        with self.qc.if_test((self.cr_z1, 3)): self.qc.x(self.qr_data[1]) # Correct Q1

        # Block 2
        with self.qc.if_test((self.cr_z2, 1)): self.qc.x(self.qr_data[3]) # Correct Q3
        with self.qc.if_test((self.cr_z2, 2)): self.qc.x(self.qr_data[5]) # Correct Q5
        with self.qc.if_test((self.cr_z2, 3)): self.qc.x(self.qr_data[4]) # Correct Q4

        # Block 3
        with self.qc.if_test((self.cr_z3, 1)): self.qc.x(self.qr_data[6]) # Correct Q6
        with self.qc.if_test((self.cr_z3, 2)): self.qc.x(self.qr_data[8]) # Correct Q8
        with self.qc.if_test((self.cr_z3, 3)): self.qc.x(self.qr_data[7]) # Correct Q7

        # --- Correcting Phase Flips (Z) ---
        # Syn '01' (dec 1) -> Error on Block 1 (Anc6 flipped, Anc7 stable)
        # Syn '10' (dec 2) -> Error on Block 3 (Anc6 stable, Anc7 flipped)
        # Syn '11' (dec 3) -> Error on Block 2 (Both flipped)

        with self.qc.if_test((self.cr_x, 1)): self.qc.z(self.qr_data[0]) # Fix Block 1
        with self.qc.if_test((self.cr_x, 2)): self.qc.z(self.qr_data[6]) # Fix Block 3
        with self.qc.if_test((self.cr_x, 3)): self.qc.z(self.qr_data[3]) # Fix Block 2

        self.qc.barrier(label='Correction Done')

    def decode_and_verify(self):
        """
        Uncomputes the encoding. If successful, data returns to |000000000>.
        """
        # Inverse Bit-Flip (CNOTs reversed)
        self.qc.cx(self.qr_data[6], self.qr_data[8]); self.qc.cx(self.qr_data[6], self.qr_data[7])
        self.qc.cx(self.qr_data[3], self.qr_data[5]); self.qc.cx(self.qr_data[3], self.qr_data[4])
        self.qc.cx(self.qr_data[0], self.qr_data[2]); self.qc.cx(self.qr_data[0], self.qr_data[1])

        # Inverse Phase-Flip (H -> CNOT)
        self.qc.h(self.qr_data[0]); self.qc.h(self.qr_data[3]); self.qc.h(self.qr_data[6])
        self.qc.cx(self.qr_data[0], self.qr_data[6])
        self.qc.cx(self.qr_data[0], self.qr_data[3])

        # Measure final state
        self.qc.measure(self.qr_data, self.cr_out)

        return self.qc


In [19]:
# Cell 3: Running a Test Case (Y-Error)
# We choose a Y-error because it is a combined X and Z error.
# The code should detect BOTH and correct BOTH.

shor = ShorCode()
shor.encode()

# Inject Y error on Qubit 0 (Topmost qubit)
print("Injecting Y error on Qubit 0...")
shor.inject_error('Y', 0)

shor.measure_syndromes()
shor.apply_correction()
circuit = shor.decode_and_verify()

# Run Simulation
backend = AerSimulator()
result = backend.run(circuit, shots=1000).result()
counts = result.get_counts()

# We look for the 'readout' register (the 9 bits on the left)
# A successful correction means the data register is '000000000'
print("\nSimulation Results (Raw Counts):")
print(counts)

# Calculate Success Rate
success_count = 0
for outcome, count in counts.items():
    # outcome string format: "readout syn_x syn_z3 syn_z2 syn_z1"
    # We only care about the readout part (first 9 bits)
    readout_part = outcome.split(" ")[0]
    if readout_part == '000000000':
        success_count += count

print(f"\nCorrected {success_count}/1000 shots.")
print(f"Fidelity: {success_count/1000:.4f}")

# Optional: Draw the circuit to verify structure
# circuit.draw('mpl')


Injecting Y error on Qubit 0...

Simulation Results (Raw Counts):
{'000000000 01 00 00 01': 1000}

Corrected 1000/1000 shots.
Fidelity: 1.0000
